# Socio-economic Profiles

This code creates the socio-economic profile data for the San Francisco Planning Department's Neighborhood Socio-Economic Profiles. Socio-economic profiles data is derived from the [American Community Survey](https://www.census.gov/programs-surveys/acs) 5-year data and the [Decennial Census](https://www.census.gov/data/developers/data-sets/decennial-census.html) and is created annually by the Planning Department. Tract level socio-economic data from is combined at the neighborhood for the City of San Francisco. This code is based off methods created by Michael Webster, Jason Sherba, and others. Run the notebook to:

- Download DEC and ACS data using the Census API
- Calculate socio-economic profiles data
- Export data to csv
- Visualize categories
- Aggregate multi-year data


## Import packages

In [1]:
# base libraries
import requests, json, os
import pandas as pd
import numpy as np
from collections import defaultdict

import warnings
warnings.filterwarnings("ignore")

## Read in the attributes lookup table

In [2]:
attribute_df = pd.read_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/lookup_tables/0_attribute_lookup - master - dec.csv', dtype=str)
attribute_df = attribute_df.fillna(np.nan).replace([np.nan], [None])
#attribute_df = attribute_df[attribute_df['attribute_name'].str.contains('Mortgage')]

In [3]:
acs_attributes = attribute_df['acs_attribute_id'].tolist() + attribute_df[attribute_df['acs_base_id'].notnull()]['acs_base_id'].tolist()
sf3_attributes = attribute_df[attribute_df['sf3_attribute_id'].notnull()]['sf3_attribute_id'].tolist() + attribute_df[attribute_df['sf3_base_id'].notnull()]['sf3_base_id'].tolist()

acs_attributes = [a for a in acs_attributes if str(a) != 'None']
sf3_attributes = [a for a in sf3_attributes if str(a) != 'None']

In [4]:
sf3_attributes

['P008003, P008004, P008005, P008006, P008007, P008042, P008043, P008044, P008045, P008046',
 'P008021, P008022, P008023, P008024, P008025, P008026, P008027, P008060, P008061, P008062, P008063, P008064, P008065, P008066',
 'P008028, P008029, P008030, P008031, P008032, P008067, P008068, P008069, P008070, P008071',
 'P008008, P008009, P008010, P008011, P008012, P008013, P008014, P008015, P008016, P008017, P008018, P008019, P008020, P008047, P008048, P008049, P008050, P008051, P008052, P008053, P008054, P008055, P008056, P008057, P008058, P008059',
 'P008033, P008034, P008035, P008036, P008037, P008038, P008039, P008040, P008072, P008073, P008074, P008075, P008076, P008077, P008078, P008079',
 'P037015, P037016, P037017, P037018, P037032, P037033, P037034, P037035',
 'P148008, P148009, P148016, P148017',
 'P037003, P037004, P037005, P037006, P037007, P037008, P037009, P037010, P037011, P037020, P037021, P037022, P037023, P037024, P037025, P037026, P037027, P037028',
 'P148003, P148004, P1

### Read in ACS attributes and set formatting

In [5]:
# In[]:   
acs_group_length = 6

acs_attribute_ids = []
acs_group_ids = []
acs_attribute_names = []

temp_df_main = attribute_df[attribute_df['acs_attribute_id'].notnull()]
temp_df_base = attribute_df[attribute_df['acs_base_id'].notnull()]

#print(attribute_ids_extracted)
ls = ['A','B','C','D','E','F','G','H','I']
for attribute_id in acs_attributes:
    #print(attribute_id)
    if len(attribute_id) > acs_group_length:
        
        acs_attribute_ids.extend(attribute_id.split(", "))
        
        if temp_df_main[temp_df_main['acs_attribute_id'].str.contains(attribute_id)]['acs_race'].any() or \
            temp_df_base[temp_df_base['acs_base_id'].str.contains(attribute_id)]['acs_race'].any():
            rs = attribute_id.split(", ")
            acs_attribute_ids.extend(list(set([x[:acs_group_length] + l + x[acs_group_length:] for l in ls for x in rs])))
        
    elif len(attribute_id) <= acs_group_length:
        acs_group_ids.append(attribute_id)
        if temp_df_main[temp_df_main['acs_attribute_id'].str.contains(attribute_id)]['acs_race'].any() or \
            temp_df_base[temp_df_base['acs_base_id'].str.contains(attribute_id)]['acs_race'].any():
            rs = attribute_id.split(", ")
            acs_group_ids.extend(list(set([x[:acs_group_length] + l + x[acs_group_length:] for l in ls for x in rs])))
    
        
acs_attribute_ids = list(set([x+"E" for x in acs_attribute_ids]))
acs_attribute_ids = acs_attribute_ids + acs_group_ids

acs_attribute_ids.sort()
acs_attribute_ids = [x.strip() for x in acs_attribute_ids]

#acs_attribute_ids

### Read in Decennial Census Summary File 3 (SF3) attributes and set formatting

In [6]:
temp_df = attribute_df[attribute_df['sf3_attribute_id'].notnull()]
temp_df_base = attribute_df[attribute_df['sf3_base_id'].notnull()]
sf3_attribute_ids = []
sf3_group_ids = []
#print(attribute_ids_extracted)
ls = ['A','B','C','D','E','F','G','H','I']
for attribute_id in sf3_attributes:
    
    if 'CT' in attribute_id:
        sf3_group_length = 6
    else:
        sf3_group_length = 4
    
    if len(attribute_id) > sf3_group_length:
        sf3_attribute_ids.extend(attribute_id.split(", "))
        
        if temp_df[temp_df['sf3_attribute_id'].str.contains(attribute_id)]['sf3_race'].any() or \
            temp_df_base[temp_df_base['sf3_base_id'].str.contains(attribute_id)]['sf3_race'].any():
            rs = attribute_id.split(", ")
            sf3_attribute_ids.extend(list(set([x[:sf3_group_length] + l + x[sf3_group_length:] for l in ls for x in rs])))
        
    elif len(attribute_id) <= sf3_group_length:
        sf3_group_ids.append(attribute_id)
        if temp_df[temp_df['sf3_attribute_id'].str.contains(attribute_id)]['sf3_race'].any() or \
            temp_df_base[temp_df_base['sf3_base_id'].str.contains(attribute_id)]['sf3_race'].any():
            rs = attribute_id.split(", ")
            sf3_group_ids.extend(list(set([x[:sf3_group_length] + l + x[sf3_group_length:] for l in ls for x in rs])))
    
        
sf3_attribute_ids = sf3_attribute_ids + sf3_group_ids

sf3_attribute_ids.sort()
sf3_attribute_ids = [x.strip() for x in sf3_attribute_ids]

#sf3_attribute_ids

In [7]:
acs_attribute_ids = list(set(acs_attribute_ids))
sf3_attribute_ids = list(set(sf3_attribute_ids))

## Retrieve data from Census API
All socio-economic data comes from the Census ACS 5-year estimates and is available at the tract level through the census API. API documentation and data for the 2023 ACS data and previous years is available (https://www.census.gov/data/developers/data-sets/acs-5year.html). DEC 2000, 2010, and 2020 data is also available (https://www.census.gov/data/developers/data-sets/decennial-census.2000.html#list-tab-517985795)

#### Build Census API URL and Make Query
The code below builds the URL for the census API call to get relevant attribute data at the tract level for San Francisco County. The Census API accepts up to 50 attributes at a time. Therefore the attribute list is first grouped into sublists of 45 attribute IDs. An API call is. Below define:
- Tract code is '*' to collect all tracts
- State code is '06' for CA
- County code is '075' for San Francisco County
- Attributes are defined by the attribute id list and includes all relevant attributes for the socio-economic data calcs

In [8]:
# function builds the api URL from tract_code, state_code, county_code, and attribute ids. Requires Census key (free)
def build_census_url(year, survey, tablename, county_code, state_code, tract_code, level, attribute_ids, my_census_key):
    attributes = ','.join(attribute_ids)
    if level == 'tract':
        census_url = r'https://api.census.gov/data/{}/{}/{}?get={}&for=tract:{}&in=state:{}&in=county:{}&key={}'.format(year, survey, tablename, attributes, tract_code, state_code, county_code, my_census_key)
    else: 
        census_url = r'https://api.census.gov/data/{}/{}/{}?get={}&for=county:{}&in=state:{}&key={}'.format(year, survey, tablename, attributes, county_code, state_code, my_census_key)
    return census_url

In [9]:
# function makes a single api call and collects results in a pandas dataframe
def make_census_api_call(census_url):
    # make API call to Census
    resp = requests.get(census_url)
    
    if resp.status_code != 200:
        # this means something went wrong
        resp.raise_for_status()
       
    # retrieve data as json and convert to Pandas Dataframe
    data = resp.json()
    headers = data.pop(0)
    
    df = pd.DataFrame(data, columns=headers)

    # convert values that are not state, county, or tract to numeric type
    try:
        cols=[i for i in df.columns if i not in ["state","county","tract"]]
    except:
        cols=[i for i in df.columns if i not in ["state","county"]]

    for col in cols:
        df[col]=pd.to_numeric(df[col])
        
    return df


In [ ]:
# set geo variables for api call
state_code = "06"
county_code = "075"
tract_code = "*"
level = 'tract'
key = '7a15f0a0073725077c4c045ea378e5947a25e9b9'

# split attributes into groups of 45, run a census query for each, merge outputs into a single df
df = pd.DataFrame({'ID': acs_attribute_ids})
df['Key'] = df['ID'].str[:6]
split_attribute_ids = df.groupby('Key')['ID'].apply(list).tolist()
split_attribute_ids = [item for sublist in split_attribute_ids for item in sublist]

#split_attribute_ids = [acs_attribute_ids[i:i+45] for i in range(0, len(acs_attribute_ids), 45)]
#print(split_attribute_ids)
YEARS = range(2010,2025)
for year in YEARS:
    df= None
    first = True
    num = 0
    for ids in split_attribute_ids:
        ids = [ids]
        census_url = build_census_url(year, 'acs', 'acs5', county_code, state_code, tract_code, level, ids, key)
        print(census_url)
        
        try:
    
            returned_df = make_census_api_call(census_url)
            num+=1
            if first:
                df = returned_df
                first = False
                
            else:
                returned_df = returned_df.drop(columns=['state'])
                df = pd.merge(df, returned_df, on=level, how='left', suffixes=(f'_x{num}', f'_y{num}'))
                
        except Exception as e:
            print(e)
            
    try:
        df = df.loc[:, ~df.columns.str.contains('^county')]
    except:
        pass
       
    df_numerics_only = df.select_dtypes(include=np.number)
    df_numerics_only
    df_numerics_only[df_numerics_only < 0] = None
    
    acs_df = pd.concat([df.select_dtypes(include=object),df_numerics_only],axis=1)
    acs_df.to_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/census_raw/acs%sdf.csv' %(year))

https://api.census.gov/data/2010/acs/acs5?get=B01001_009E&for=tract:*&in=state:06&in=county:075&key=7a15f0a0073725077c4c045ea378e5947a25e9b9
https://api.census.gov/data/2010/acs/acs5?get=B01001H_001E&for=tract:*&in=state:06&in=county:075&key=7a15f0a0073725077c4c045ea378e5947a25e9b9
https://api.census.gov/data/2010/acs/acs5?get=B01001_038E&for=tract:*&in=state:06&in=county:075&key=7a15f0a0073725077c4c045ea378e5947a25e9b9
https://api.census.gov/data/2010/acs/acs5?get=B01001_023E&for=tract:*&in=state:06&in=county:075&key=7a15f0a0073725077c4c045ea378e5947a25e9b9
https://api.census.gov/data/2010/acs/acs5?get=B01001_025E&for=tract:*&in=state:06&in=county:075&key=7a15f0a0073725077c4c045ea378e5947a25e9b9
https://api.census.gov/data/2010/acs/acs5?get=B01001_029E&for=tract:*&in=state:06&in=county:075&key=7a15f0a0073725077c4c045ea378e5947a25e9b9
https://api.census.gov/data/2010/acs/acs5?get=B01001_011E&for=tract:*&in=state:06&in=county:075&key=7a15f0a0073725077c4c045ea378e5947a25e9b9
https://api.

In [ ]:
# set geo variables for api call
key = '7a15f0a0073725077c4c045ea378e5947a25e9b9'

state_code = "06"
county_code = "075"
tract_code = "*"
level = 'tract'
# split attributes into groups of 45, run a census query for each, merge outputs into a single df
df = pd.DataFrame({'ID': sf3_attribute_ids})
df['Key'] = df['ID'].str[:6]
split_attribute_ids = df.groupby('Key')['ID'].apply(list).tolist()
split_attribute_ids = [item for sublist in split_attribute_ids for item in sublist]

#split_attribute_ids = [sf3_attribute_ids[i:i+45] for i in range(0, len(sf3_attribute_ids), 45)]
#print(split_attribute_ids)
df= None
first = True
num=0
year = 2000
for ids in split_attribute_ids:
    #print(ids)
    ids = [ids]
    census_url = build_census_url(year, 'dec', 'sf3', county_code, state_code, tract_code, level, ids, key)
    print(census_url)
    try:

        returned_df = make_census_api_call(census_url)
        num+=1
        if first:
            df = returned_df
            first = False
            
        else:
            returned_df = returned_df.drop(columns=['state'])
            df = pd.merge(df, returned_df, on=level, how='left', suffixes=(f'_x{num}', f'_y{num}'))
            
    except Exception as e:
        print(e)
        
try:
    df = df.loc[:, ~df.columns.str.contains('^county')]
except:
    pass
   
df_numerics_only = df.select_dtypes(include=np.number)
df_numerics_only
df_numerics_only[df_numerics_only < 0] = None

sf32000df = pd.concat([df.select_dtypes(include=object),df_numerics_only],axis=1)
sf32000df.to_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/census_raw/sf32000df.csv')

## Prepare Lookup Dictionaries and Helper Functions

### Calculating Medians
To calculate median values of aggregated geographies you cannot use the mean of component geographies. Instead a statistical approximation of the median must be calculated from range tables. 

Range variables in the ACS have a unique ID like any other Census variable. They represent the amount of a variable within a select range. e.g. number of households with household incomes between $45000-50000. Range variable ID's and range information is stored in the median_ranges.csv file in the repository. These range variables and ranges are needed for calculating the median at the neighborhood level. 

The below function calculates a median based on range data. This method follows the offical ACS documentation for [calculating a median](https://www.dof.ca.gov/Forecasting/Demographics/Census_Data_Center_Network/documents/How_to_Recalculate_a_Median.pdf)


In [ ]:
median_dict = {"Median Year Structure Built":'median_year_structure_built',
               "Median Year Moved In Owner":'median_year_moved_owner',
               "Median Year Moved In Renter":'median_year_moved_renter',
               "Median Rent":'median_rent',
               "Median Contract Rent":'median_rent_contract',
               "Median Rent as % of Household Income":'median_rent_percent_of_income',
               "Median Household Income":'median_household_income',
               "Median Household Income by Race":'median_household_income_by_race',
               "Median Family Income":'median_family_income',
               "Median Home Value":'median_home_value'}

# define median helper function
def calc_median(tract_df, range_df, median_to_calc, ids, year):

    # subset range df for current median variable to calc
    range_df = range_df[range_df['name']==median_to_calc]
    if year == 2000 or 'income' in median_to_calc:
        range_min_col = 1
        range_max_col = 2
    else:
        l = range_df.iloc[0][2:].values
        
        range_min_col = range_df.iloc[0][2:].tolist().index(l[l <= year][-1]) + 1
        try:
            range_max_col = range_df.iloc[0][2:].tolist().index(l[l > year][0]) + 1
        except:
            range_max_col = range_df.iloc[0][2:].tolist().index(max(range_df.iloc[0][2:].tolist())) + 2
            
    if range_min_col == range_max_col:
        print("HUGEEEE ERRRORRRR")
    range_df = range_df.iloc[1:]
    range_df = range_df.rename(columns={str(range_min_col):'range_start',
                                        str(range_max_col):'range_end'}, inplace=False)
    # sort dataframe low to high by range start column
    range_df = range_df.sort_values(by=['range_start'])
    
    # remove nan rwos
    range_df = range_df[range_df['range_start'].notnull()]
    
    #range_df =  range_df[['name','id','range_start','range_end']]
    range_df['households']=0
    range_df['cumulative_total']=0
    
    # calculate households as sum of tract level households for each row based on range id
    if 'by_race' in median_dict[attribute_name.split(',')[0]]: 
        ids.sort()
        range_df['id'] = ids
    #print(range_df['id'])
    
    range_df['households'] = range_df.apply(lambda row : tract_df[row['id']].sum(), axis = 1)
    
    # calculate the cumulative total of households
    range_df['cumulative_total'] = range_df['households'].cumsum()
    
    # calculate total households and return 0 if total households is 0
    total_households = range_df['households'].sum()
    
    # if total households is 0 set median to 0
    if total_households == 0:
        return 0
    
    # calculate midpoint
    midpoint = total_households/2

    # if midpoint is below first range return median as end of first range value
    if midpoint <= range_df['cumulative_total'].min():
        new_median = range_df['range_end'].min()
        return new_median
    
    # if midpoint is above last range set median to end of last range value
    if midpoint >= range_df['cumulative_total'].max():
        new_median = range_df['range_end'].max()
        return new_median
    
    less_midpoint_df = range_df[range_df['cumulative_total']<midpoint]
    
    # get the single row containing the range just below the mid range by getting the row with the max range start from the subsetted median df
    range_below_mid_range_df = less_midpoint_df[less_midpoint_df['range_start'] == less_midpoint_df['range_start'].max()]
    
    # get the cumulative total value for the first row of the range below mid range dictionary
    total_hh_previous_range = range_below_mid_range_df['cumulative_total'].iloc[0]
    hh_to_mid_range = midpoint - total_hh_previous_range
    
    # extract rows above midrange by subsetting median df for rows with cumulative total grearter than midpoint.
    greater_midpoint_df = range_df[range_df['cumulative_total']>midpoint]
    
    # get the single row containing the mid range by getting the row with the min range start from the subsetted median df
    mid_range_df = greater_midpoint_df[greater_midpoint_df['range_start'] == greater_midpoint_df['range_start'].min()]
    
    # get the households value for the first row of the mid range dictionary
    hh_in_mid_range = mid_range_df['households'].iloc[0]
    
    # calculate proportion of number of households in the mid range that would be needed to get to the mid-point
    prop_of_hh = hh_to_mid_range/hh_in_mid_range
    
    # calculate width of the mid range
    width = (mid_range_df['range_end'].iloc[0]-mid_range_df['range_start'].iloc[0])+1
    
    # apply proportion to width of mid range
    prop_to_width = prop_of_hh*width
    beginning_of_mid_range = mid_range_df['range_start'].iloc[0]
    
    # calculate new median
    new_median = beginning_of_mid_range + prop_to_width
    
    return new_median


## Define functions for calculating socio-economic data
The `calc_socio_economic_data` function takes tract level data from the API call and the tract/neighborhood lookup dictionary. This function creates all of the socio-economic data calcs and returns a dictionary. The calcs in this function are derived from the [Data_Items_and_Sources.xlsx](https://github.com/jsherba/socio-economic-profiles/raw/main/Data_Items_And_Sources_2019.xlsx) file developed by Michael Webster

In [ ]:
def check_attribute_ids(available_attribute_ids, attribute_ids):
    ATTRIBUTE_IDS = []
    for i in attribute_ids:
        if i in available_attribute_ids:
            ATTRIBUTE_IDS.append(i)
    return ATTRIBUTE_IDS

In [ ]:
# function runs all calcs for each neighborhood
def calc_socio_economic_data(df, tract_lookup, all_calc_data, all_calc_data_by_tract, \
                             attribute_name, \
                             attribute_ids, base_ids, treatment, year):
    # nb_name = 'Western Addition/Fillmore Community Boundary'
    # tracts = tract_lookup[nb_name]
    
    for nb_name, tracts in tract_lookup.items():
        #print(nb_name, tracts)
        
        # extract attribute information for tracks associated with a neighborhood
        tract_df = df[df['tract'].isin(tracts)]
        available_attribute_ids = tract_df.columns
        # build dictionary with all stats for a neighborhood
        all_calc_data_nb = all_calc_data[nb_name]
        
        attribute_ids.sort()
        attribute_ids = check_attribute_ids(available_attribute_ids, attribute_ids)
        attribute = tract_df[attribute_ids]
        #print(treatment)
        #print(attribute)
        #print(attribute_ids,base_ids)
        if treatment == 'as is' and base_ids:
            base = tract_df[base_ids]
            null_idx = attribute[pd.isnull(attribute).any(axis=1)].index.tolist() + base[pd.isnull(base).any(axis=1)].index.tolist()
            attribute = attribute[attribute.index.isin(null_idx) == False]
            base = base[base.index.isin(null_idx) == False]   
            all_calc_data_nb['%s' %attribute_name] = attribute.sum().sum()/base.sum().sum()
        
        elif treatment == 'wa' and base_ids:
            base = tract_df[base_ids]
            null_idx = attribute[pd.isnull(attribute).any(axis=1)].index.tolist() + base[pd.isnull(base).any(1)].index.tolist()
            #print(null_idx)
            attribute = attribute[attribute.index.isin(null_idx) == False]
            base = base[base.index.isin(null_idx) == False]    
            #print(attribute,base)
            all_calc_data_nb['%s' %attribute_name] = (attribute.values *base.values).sum() / base.sum().sum()
            
        elif treatment == 'as is' and not base_ids:
            null_idx = attribute[pd.isnull(attribute).any(axis=1)].index.tolist()
            attribute = attribute[attribute.index.isin(null_idx) == False]
            all_calc_data_nb['%s' %attribute_name] = attribute.sum().sum()

        elif treatment == 'median':
            #print(attribute_name,attribute_ids)
            all_calc_data_nb['%s' %attribute_name] = calc_median(tract_df, range_df, \
                                                                 median_dict[attribute_name.split(',')[0]], attribute_ids, year)
               

        # try:
            
        #     print(all_calc_data_nb['%s' %attribute_name])
            
        # except:
            
        #     pass
        
    for tract_name in all_tracts:
        #print(tract_name)
        tract_df = df[df['tract'].isin([tract_name])]

        all_calc_data_nb = all_calc_data_by_tract[tract_name]
        
        if treatment == 'as is' and base_ids:
                all_calc_data_nb['%s' %attribute_name] = tract_df[attribute_ids].sum().sum()/tract_df[base_ids].sum().sum()

        elif treatment == 'as is' and not base_ids:
            all_calc_data_nb['%s' %attribute_name] = tract_df[attribute_ids].sum().sum()

        elif treatment == 'median':
            #print(range_df, attribute_name)
            all_calc_data_nb['%s' %attribute_name] = calc_median(tract_df, range_df, median_dict[attribute_name.split(' by')[0]], attribute_ids, year)

        # try:
        #     print(all_calc_data_nb['%s' %attribute_name])
        # except:
        #     pass
    #print('all_calc_data_nb')
    #print(all_calc_data_nb)
    #print('all_calc_data')
    #print(all_calc_data)
    #print('all_calc_data_by_tract')
    #print(all_calc_data_by_tract)

    return all_calc_data, all_calc_data_by_tract


## Calculate Socioeconomic Profiles

### Run Socioeconomic Profiles Calcs

In [ ]:
download_path = r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/downloads'

In [ ]:
attribute_df['attribute_name'].tolist()

In [ ]:
acs_race ={'A':'White Alone', 'B':'Black or African American Alone', 'C':'American Indian and Alaska Native Alone',
           'D':'Asian Alone', 'E': 'Native Hawaiian and Other Pacific Islander Alone',
           'F': 'Some Other Race Alone', 'G': 'Two or More Races', 'H': 'White Alone, Not Hispanic or Latino',
           'I': 'Hispanic or Latino'}
sf3_race = {'A':'White Alone', 'B':'Black or African American Alone', 'C':'American Indian and Alaska Native Alone',
           'D':'Asian Alone', 'E': 'Native Hawaiian and Other Pacific Islander Alone',
           'F': 'Some Other Race Alone', 'G': 'Two or More Races', 'H': 'Hispanic or Latino',
           'I': 'White Alone, Not Hispanic or Latino'}
geo_summary_variable = 'Neighborhood'

YEARS = [2000, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

geo_lookup_df = pd.read_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/lookup_tables/geo_lookup_2000to2020.csv', dtype=str)

for year in YEARS:    

    if year <= 2009:
        geo_year = '2000'
        survey = 'dec'
        table_name = 'sf3'
        label = 'DEC_SF3'
        ext = ''
    elif year >= 2010 and year <= 2019:
        geo_year = '2010'
        survey = 'acs'
        table_name = 'acs'
        label = 'ACS5YR'
        ext = 'E'
    elif year >= 2020:
        geo_year = '2020'
        survey = 'acs'
        table_name = 'acs'
        label = 'ACS5YR'
        ext = 'E'
    
    dict_race = {}
    exec('dict_race = %s_race' %table_name)

    # import median tables from median_ranges csv and add empty columns for rows 'households and 'cumulative_totals'
    range_df = pd.read_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/lookup_tables/%s_median_ranges.csv' %table_name)
    
    print(table_name, year)
    df = pd.read_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/census_raw/%s%sdf.csv' %(table_name, year))
    for i in df['tract'].index:
        j = str(df.loc[i,'tract'])
        j = '0' + j
        while len(j) < 6:
            j = j + '0'
        df.loc[i,'tract'] = j
    
    # import geo_lookup csv
    geo_lookup_df_yyyy = geo_lookup_df[geo_lookup_df['year'] == geo_year]
    neighborhoods = list(set(geo_lookup_df_yyyy.neighborhood.tolist()))
    neighborhoods.sort()
    neighborhoods.remove('San Francisco')
    #print(geo_lookup_df_yyyy)
    tract_nb_lookup = defaultdict(list)
    all_tracts = list(set(df['tract'].tolist()))
    
    # create tract lookup dictionary for neighborhoods
    for i, j in zip(geo_lookup_df_yyyy['neighborhood'], geo_lookup_df_yyyy['tractid']):
        if len(j) == 5:
            tract_nb_lookup[i].append('0'+j)
        else:
            tract_nb_lookup[i].append(j)
        
    #tract_nb_lookup["sf"]= all_tracts
    # create tract lookup dictionary for supervisor districts
    first_4 = list(tract_nb_lookup.items())
    #first_4
    
    geo_summary_variable = 'neighborhood'
    
    tract_lookup = tract_nb_lookup
    #print(tract_lookup)
    
        ## Calculate Socioeconomic Profiles
        
    all_calc_data = defaultdict(dict) 
    all_calc_data_by_tract = defaultdict(dict)
    attribute_names = []
    
    for j in attribute_df.index:
        #print(j)
        attribute_name = attribute_df.loc[j, 'attribute_name']
        
        attribute_id = [attribute_df.loc[j, '%s_attribute_id' %table_name]]

        if attribute_id == [None]:
            continue
        
        base_id = [attribute_df.loc[j, '%s_base_id' %table_name]]
        
        race = attribute_df.loc[j, '%s_race' %table_name]
        treatment = attribute_df.loc[j, '%s_treatment' %table_name]
        
        attribute_ids = [x.strip().split(", ") for x in attribute_id][0]
        attribute_ids = list(set([x+ext for x in attribute_ids]))
        
        try:
            base_ids = [x.strip().split(", ") for x in base_id][0]
            base_ids = list(set([x+ext for x in base_ids]))
        except:
            base_ids = None
        
        #print(attribute_name, attribute_id, race, base_ids, attribute_ids, treatment)

        
        if race is not None: 
            if 'CT' in attribute_ids[0] or attribute_ids[0].startswith('B') or attribute_ids[0].startswith('C'):
                group_length = 6
            else:
                group_length = 4
                
            for r in dict_race.keys():
                #print(r)
                attribute_name_by_r = attribute_name + ', '+ dict_race[r]
                
                attribute_ids_by_r = list(set([x[:group_length] + l + x[group_length:] for l in r for x in attribute_ids]))
                
                if base_ids is not None:
                    base_ids_by_r = list(set([x[:group_length] + l + x[group_length:] for l in r for x in base_ids]))
                else:
                    base_ids_by_r = None
                
                #print(attribute_name_by_r, attribute_ids_by_r, base_ids_by_r, treatment)
                attribute_names.append(attribute_name_by_r)
                try:
                    #print(attribute_name_by_r, attribute_ids_by_r, base_ids_by_r, treatment, year)
                    all_calc_data, all_calc_data_by_tract = calc_socio_economic_data(df, tract_lookup, all_calc_data, all_calc_data_by_tract,
                                                                             attribute_name_by_r, attribute_ids_by_r, 
                                                                             base_ids_by_r, treatment, year)
                    
                except Exception as e:
                    print(e)
                    print(attribute_name, attribute_ids, base_ids)

        elif treatment == 'loop':
            idx = 0
            for r in dict_race.keys():
                
                if r == 'H' or r == 'I':
                    print('skip')
                else:
                    #print(r)
                    attribute_name_by_r = attribute_name + ', '+ dict_race[r]
                    
                    attribute_ids_by_r = [attribute_ids[idx]] #list(set([x[:group_length] + l + x[group_length:] for l in r for x in attribute_ids]))
                    idx += 1
                    if base_ids is not None:
                        base_ids_by_r = base_ids #list(set([x[:group_length] + l + x[group_length:] for l in r for x in base_ids]))
                    else:
                        base_ids_by_r = None
                    
                    #print(attribute_name_by_r, attribute_ids_by_r, base_ids_by_r, treatment)
                    attribute_names.append(attribute_name_by_r)
                    treatment = 'as is'
                    try:
                        all_calc_data, all_calc_data_by_tract = calc_socio_economic_data(df, tract_lookup, all_calc_data, all_calc_data_by_tract,
                                                                                 attribute_name_by_r, attribute_ids_by_r, 
                                                                                 base_ids_by_r, treatment, year)
                    except Exception as e:
                        print(e)
                        print(attribute_name, attribute_ids, base_ids)

                
        else:
            attribute_names.append(attribute_name)
            #print(attribute_name, attribute_ids, base_ids)
            try:
                all_calc_data, all_calc_data_by_tract = calc_socio_economic_data(df, tract_lookup, \
                                                                         all_calc_data, all_calc_data_by_tract, attribute_name, \
                                                                         attribute_ids, base_ids, treatment, year)
            except Exception as e:
                print(e)
                print(attribute_name, attribute_ids, base_ids)

        
    
    # code segment to account for alternates
    #def remove_alternate_text(x):
    #    return x.replace(', alternate','')
        
    
    cat_dict = dict(zip(attribute_df.attribute_name,attribute_df.category_dag))
    #print(df_all_calcs['Attribute'])

    df_all_calcs = pd.DataFrame.from_dict(all_calc_data).reset_index()
    df_all_calcs.rename(columns = {'index':'Attribute'}, inplace = True) 
    
    # Alternates:
    print(df_all_calcs.Attribute)
    #df_all_calcs['Attribute'] = df_all_calcs['Attribute'].apply(lambda x: remove_alternate_text(x))
    print(df_all_calcs)
    #df_all_calcs = df_all_calcs.fillna(0)
    #zero_sum_attributes = df_all_calcs[df_all_calcs.sum(axis=1) == 0].index
    #df_all_calcs.loc[zero_sum_attributes,neighborhoods] = np.nan
    #df_all_calcs.loc[zero_sum_attributes,'San Francisco'] = np.nan
    
    # Sort to prioritize rows with non-null 'Value'
    df_all_calcs = df_all_calcs.sort_values(by='San Francisco', na_position='last')
    
    # Drop duplicate attributes, keeping the first occurrence (non-null prioritized)
    print(df_all_calcs)
    df_all_calcs = df_all_calcs.drop_duplicates(subset='Attribute', keep='first')
    
    # Reset index if needed
    df_all_calcs = df_all_calcs.reset_index(drop=True)

    df_all_calcs['Category'] = df_all_calcs['Attribute'].map(cat_dict)
    temp = df_all_calcs[df_all_calcs['Category'].isnull()]

    if len(temp['Attribute'].str.split(', ',n=1,expand=True)):    
        temp['Race delimiter'] = None
        temp[['Attribute 2','Race delimiter 2']] = temp['Attribute'].str.split(', ',n=1,expand=True)
        temp[['Attribute','Race delimiter']] = temp['Attribute'].str.split(' by',n=1,expand=True)
        df_all_calcs['Category'] = df_all_calcs['Category'].fillna(temp['Attribute'].map(cat_dict))
    
    column_order = ['Category', 'Attribute', 'San Francisco'] + neighborhoods
    df_all_calcs = df_all_calcs[column_order]
    
    df_all_calcs_by_tract = pd.DataFrame.from_dict(all_calc_data_by_tract).reset_index()
    df_all_calcs_by_tract.rename(columns = {'index':'Attribute'}, inplace = True) 
    #df_all_calcs_by_tract['Attribute'] = df_all_calcs_by_tract['Attribute'].apply(lambda x: remove_alternate_text(x))

    df_all_calcs_by_tract['Category'] = df_all_calcs_by_tract['Attribute'].map(cat_dict)
    
    temp = df_all_calcs_by_tract[df_all_calcs_by_tract['Category'].isnull()]
    if len(temp['Attribute'].str.split(', ',n=1,expand=True)):
        temp['Race delimiter'] = None
        temp[['Attribute 2','Race delimiter 2']] = temp['Attribute'].str.split(', ',n=1,expand=True)
        temp[['Attribute','Race delimiter']] = temp['Attribute'].str.split(' by',n=1,expand=True)
        df_all_calcs_by_tract['Category'] = df_all_calcs_by_tract['Category'].fillna(temp['Attribute'].map(cat_dict))
        
    df_numerics_only = df_all_calcs_by_tract.select_dtypes(include=np.number)
    
    # Sort to prioritize rows with non-null 'Value'
    #zero_sum_attributes = df_all_calcs_by_tract[df_all_calcs_by_tract.sum(axis=1) == 0].index
    #df_all_calcs_by_tract.loc[zero_sum_attributes,df_numerics_only.columns] = np.nan
    
    try:
        df_all_calcs_by_tract = df_all_calcs_by_tract.sort_values(by='010100', na_position='last')
    except:
        df_all_calcs_by_tract = df_all_calcs_by_tract.sort_values(by='030800', na_position='last')

    # Drop duplicate attributes, keeping the first occurrence (non-null prioritized)
    df_all_calcs_by_tract = df_all_calcs_by_tract.drop_duplicates(subset='Attribute', keep='first')
    
    # Reset index if needed
    df_all_calcs_by_tract = df_all_calcs_by_tract.reset_index(drop=True)
    
    df_numerics_only = df_all_calcs_by_tract.select_dtypes(include=np.number)

    df_cat_columns = df_all_calcs_by_tract.select_dtypes(include=object)
    df_cat_columns = df_cat_columns[['Category', 'Attribute']]
    df_all_calcs_by_tract = pd.concat([df_cat_columns,df_numerics_only],axis=1)
    
    df_all_calcs = df_all_calcs.sort_values(by= ['Category', 'Attribute'], ascending = [True, True], inplace=False)

    df_all_calcs.to_csv(os.path.join(download_path,'Neighborhood'+'_'+'profiles_by_attribute_'+label+'_{}.csv'.format(year)), index=False)
    
    # transpose dataset for second geo view of dataset
    df_all_calcs_tp = df_all_calcs.T.reset_index()
    df_all_calcs_tp.columns = df_all_calcs_tp.iloc[0]
    df_all_calcs_tp = df_all_calcs_tp[1:].rename(columns={'Category': geo_summary_variable})
    df_all_calcs_tp.to_csv(os.path.join(download_path,'Neighborhood'+'_'+'profiles_by_geo_'+label+'_{}.csv'.format(year)), index=False)

    #################################### 
    df_all_calcs_by_tract = df_all_calcs_by_tract.sort_values(by= ['Category', 'Attribute'], ascending = [True, True], inplace=False)
    df_all_calcs_by_tract.to_csv(os.path.join(download_path,'Tract'+'_'+'profiles_by_attribute_'+label+'_{}.csv'.format(year)), index=False)
    
    # transpose dataset for second geo view of dataset
    df_all_calcs_tp = df_all_calcs_by_tract.T.reset_index()
    df_all_calcs_tp.columns = df_all_calcs_tp.iloc[0]
    df_all_calcs_tp = df_all_calcs_tp[1:].rename(columns={'Category': "tract"})
    df_all_calcs_tp = df_all_calcs_tp.sort_values(by=["tract"])
    df_all_calcs_tp.to_csv(os.path.join(download_path,'Tract'+'_'+'profiles_by_geo_'+label+'_{}.csv'.format(year)), index=False)

# Aggregate files

## Generate neighborhood profiles

In [ ]:
# In[]: Generate neighborhood profiles
import pandas as pd
YEARS = [2024, 2023, 2022,2021,2020,2019,2018,2017,2016,2015,2014,2013,2012,2011,2010,2000]
import os
import warnings
warnings.filterwarnings("ignore")
download_path = r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/downloads'
geo_lookup_df = pd.read_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/lookup_tables/geo_lookup_2000to2020.csv', dtype=str)
geo_lookup_df_yyyy = geo_lookup_df[geo_lookup_df['year'] == '2020']
neighborhoods = list(set(geo_lookup_df_yyyy.neighborhood.tolist()))
neighborhoods.sort()
    
for n in neighborhoods:
    
    year_columns = []
    
    for year in YEARS:    
    
        if year <= 2009:
            geo_year = '2000'
            survey = 'dec'
            table_name = 'sf3'
            label = 'DEC_SF3'
            ext = ''
        elif year >= 2010 and year <= 2019:
            geo_year = '2010'
            survey = 'acs'
            table_name = 'acs'
            label = 'ACS5YR'
            ext = 'E'
        elif year >= 2020:
            geo_year = '2020'
            survey = 'acs'
            table_name = 'acs'
            label = 'ACS5YR'
            ext = 'E'
            
        df_all_calcs = pd.read_csv(os.path.join(download_path,'neighborhood'+'_'+'profiles_by_attribute_'+label+'_{}.csv'.format(year)))
    
        try:    
            
            if year == 2024:
                
                df_all_calcs_full = df_all_calcs[['Category','Attribute',n]]
                df_all_calcs_full.rename(columns={n:str(year) + '_' + n}, inplace=True)
                
                # CALCULATE AFFORDABILITY
                ami_breaks = [30, 50, 80, 100]
                df_all_calcs_full_aff = pd.DataFrame(columns = df_all_calcs_full.columns, \
                                                        index = range(12))
                df_all_calcs_full_aff['Category'] = 'Affordability'

                c = 0
                for i in ami_breaks:

                    df_all_calcs_full_aff.loc[c, 'Attribute'] = '%s AMI Annual Income' %i
                    df_all_calcs_full_aff.loc[c+1, 'Attribute'] = '%s AMI Monthly Affordable Rent' %i
                    df_all_calcs_full_aff.loc[c+2, 'Attribute'] = '# Units at or below %s AMI Monthly Affordable Rent' %i
                    c+=3

                rent_levels = df_all_calcs_full[(df_all_calcs_full.Attribute.str.contains("Rent")) & \
                                   ((df_all_calcs_full.Attribute.str.contains(" to ")) | \
                                   (df_all_calcs_full.Attribute.str.contains(" or ")))].to_dict('records')
                max_dict = {}
                for j in rent_levels:
                    level_j = j
                    key_j = j['Attribute']
                    key_j_split = key_j.split(' ')
                    try:
                        max_dict[key_j] = int(key_j_split[3])
                    except:
                        max_dict[key_j] = 1e6

                l = list(max_dict.values())
                
            else:
                df_all_calcs = df_all_calcs[['Attribute',n]]
                df_all_calcs.rename(columns={n:str(year) + '_' + n}, inplace=True)
                df_all_calcs_full = df_all_calcs_full.merge(df_all_calcs, how='left', on = 'Attribute')
                
            year_columns.append(year)
            
        except Exception as e:
            print(n, year, e)

        try:
            median_income = float(df_all_calcs_full[df_all_calcs_full.Attribute.isin([
                                            "Median Household Income"                                                                            
                                            ])].to_dict('records')[0]['%s' %(str(year) + '_' + n)])

            income_breaks = [(i/100) * median_income for i in ami_breaks]
            breaks_dict = dict(zip(ami_breaks, income_breaks))

            #print(breaks_dict, income_breaks)
            # Affordable limits for each AMI tier:
            c = 0
            
            for i in ami_breaks:
                
                df_all_calcs_full_aff.loc[c, '%s' %(str(year) + '_' + n)] = breaks_dict[i]
                df_all_calcs_full_aff.loc[c+1, '%s' %(str(year) + '_' + n)] = breaks_dict[i]/12
                
                aff_rent = breaks_dict[i]/12

                m = [x for x in l if x <= aff_rent]
                counter = [key for key, value in max_dict.items() if value in m]
                counter = int(df_all_calcs_full[df_all_calcs_full.Attribute.isin(counter)]['%s' %(str(year) + '_' + n)].sum())

                df_all_calcs_full_aff.loc[c+2, '%s' %(str(year) + '_' + n)] = counter    
                
                c+=3

        except Exception as e:
            print(e)

    df_all_calcs_full = pd.concat([df_all_calcs_full, df_all_calcs_full_aff], axis=0)
    
    year_columns.sort()  
    columns_plus = [str(x) +'_' + n for x in year_columns]
    
    # drop na rows
    df_all_calcs_full = df_all_calcs_full.dropna(subset=columns_plus, inplace=False)
    
    columns_plus.insert(0,'Attribute')
    columns_plus.insert(0,'Category')

    df_all_calcs_full = df_all_calcs_full[columns_plus]    
    
    alternates = df_all_calcs_full[df_all_calcs_full['Attribute'].str.contains('alternate')]['Attribute'].tolist()
    #print(alternates)

    if len(alternates) > 0:
        alternates_labels = [x[:-11] for x in alternates]
        for a in alternates_labels:
            df_rows = df_all_calcs_full[df_all_calcs_full['Attribute'].str.startswith(a)]
            df_rows = df_rows.iloc[0].combine_first(df_rows.iloc[1])
            df_all_calcs_full = df_all_calcs_full[(df_all_calcs_full['Attribute'].str.startswith(a)) == False]
            df_all_calcs_full = pd.concat([df_all_calcs_full, df_rows.to_frame().T], axis=0).reset_index(drop=True)

    df_all_calcs_full = df_all_calcs_full.sort_values(by= ['Category', 'Attribute'], ascending = [True, True], inplace=False)
    
    n = n.strip().replace('/','-')
    df_all_calcs_full.to_excel(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/output/%s_neighborhood_profiles_by_attribute_2000to2024.xlsx' %n, index=False, sheet_name='census_attributes')
    df_all_calcs_full.to_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/output/%s_neighborhood_profiles_by_attribute_2000to2024.csv' %n)

## Comparative neighborhood profiles to citywide

In [ ]:

import pandas as pd
YEARS = [2024, 2023, 2022,2021,2020,2019,2018,2017,2016,2015,2014,2013,2012,2011,2010,2000]
import os
import warnings
warnings.filterwarnings("ignore")
download_path = r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/downloads'
geo_lookup_df = pd.read_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/lookup_tables/geo_lookup_2000to2020.csv', dtype=str)
geo_lookup_df_yyyy = geo_lookup_df[geo_lookup_df['year'] == '2020']
neighborhoods = list(set(geo_lookup_df_yyyy.neighborhood.tolist()))
neighborhoods.sort()
    
for n in neighborhoods:
    
    year_columns = []
    
    for year in YEARS:    
    
        if year <= 2009:
            geo_year = '2000'
            survey = 'dec'
            table_name = 'sf3'
            label = 'DEC_SF3'
            ext = ''
        elif year >= 2010 and year <= 2019:
            geo_year = '2010'
            survey = 'acs'
            table_name = 'acs'
            label = 'ACS5YR'
            ext = 'E'
        elif year >= 2020:
            geo_year = '2020'
            survey = 'acs'
            table_name = 'acs'
            label = 'ACS5YR'
            ext = 'E'
            
        df_all_calcs = pd.read_csv(os.path.join(download_path,'neighborhood'+'_'+'profiles_by_attribute_'+label+'_{}.csv'.format(year)))
        df_all_calcs_sf = pd.read_csv(os.path.join(download_path,'neighborhood'+'_'+'profiles_by_attribute_'+label+'_{}.csv'.format(year)))

        try:    
            
            if year == 2024:
                
                df_all_calcs_full = df_all_calcs[['Category','Attribute',n]]
                df_all_calcs_full.rename(columns={n:str(year) + '_' + n}, inplace=True)
                
                # CALCULATE AFFORDABILITY
                ami_breaks = [30, 50, 80, 100]
                df_all_calcs_full_aff = pd.DataFrame(columns = df_all_calcs_full.columns, \
                                                        index = range(12))
                df_all_calcs_full_aff['Category'] = 'Affordability'

                c = 0
                for i in ami_breaks:

                    df_all_calcs_full_aff.loc[c, 'Attribute'] = '%s AMI Annual Income' %i
                    df_all_calcs_full_aff.loc[c+1, 'Attribute'] = '%s AMI Monthly Affordable Rent' %i
                    df_all_calcs_full_aff.loc[c+2, 'Attribute'] = '# Units at or below %s AMI Monthly Affordable Rent' %i
                    c+=3

                rent_levels = df_all_calcs_full[(df_all_calcs_full.Attribute.str.contains("Rent")) & \
                                   ((df_all_calcs_full.Attribute.str.contains(" to ")) | \
                                   (df_all_calcs_full.Attribute.str.contains(" or ")))].to_dict('records')
                max_dict = {}
                for j in rent_levels:
                    level_j = j
                    key_j = j['Attribute']
                    key_j_split = key_j.split(' ')
                    try:
                        max_dict[key_j] = int(key_j_split[3])
                    except:
                        max_dict[key_j] = 1e6

                l = list(max_dict.values())
              
                df_all_calcs_full_sf = df_all_calcs_sf[['Category','Attribute','San Francisco']]
                df_all_calcs_full_sf.rename(columns={'San Francisco':str(year) + '_' + 'San Francisco'}, inplace=True)
                
                # CALCULATE AFFORDABILITY
                ami_breaks = [30, 50, 80, 100]
                df_all_calcs_full_sf_aff = pd.DataFrame(columns = df_all_calcs_full_sf.columns, \
                                                        index = range(12))
                df_all_calcs_full_sf_aff['Category'] = 'Affordability'

                c = 0
                for i in ami_breaks:

                    df_all_calcs_full_sf_aff.loc[c, 'Attribute'] = '%s AMI Annual Income' %i
                    df_all_calcs_full_sf_aff.loc[c+1, 'Attribute'] = '%s AMI Monthly Affordable Rent' %i
                    df_all_calcs_full_sf_aff.loc[c+2, 'Attribute'] = '# Units at or below %s AMI Monthly Affordable Rent' %i
                    c+=3

                rent_levels = df_all_calcs_full_sf[(df_all_calcs_full_sf.Attribute.str.contains("Rent")) & \
                                   ((df_all_calcs_full_sf.Attribute.str.contains(" to ")) | \
                                   (df_all_calcs_full_sf.Attribute.str.contains(" or ")))].to_dict('records')
                max_dict = {}
                for j in rent_levels:
                    level_j = j
                    key_j = j['Attribute']
                    key_j_split = key_j.split(' ')
                    try:
                        max_dict[key_j] = int(key_j_split[3])
                    except:
                        max_dict[key_j] = 1e6

                l = list(max_dict.values())
              
            else:
                df_all_calcs = df_all_calcs[['Attribute',n]]
                df_all_calcs.rename(columns={n:str(year) + '_' + n}, inplace=True)
                df_all_calcs_full = df_all_calcs_full.merge(df_all_calcs, how='left', on = 'Attribute')
                
                df_all_calcs_sf = df_all_calcs_sf[['Attribute','San Francisco']]
                df_all_calcs_sf.rename(columns={'San Francisco':str(year) + '_' + 'San Francisco'}, inplace=True)
                df_all_calcs_full_sf = df_all_calcs_full_sf.merge(df_all_calcs_sf, how='left', on = 'Attribute')
                
            year_columns.append(year)
            
        except Exception as e:
            print(n, year, e)
    
        try:
            median_income = float(df_all_calcs_full[df_all_calcs_full.Attribute.isin([
                                            "Median Household Income"                                                                            
                                            ])].to_dict('records')[0]['%s' %(str(year) + '_' + n)])

            income_breaks = [(i/100) * median_income for i in ami_breaks]
            breaks_dict = dict(zip(ami_breaks, income_breaks))

            #print(breaks_dict, income_breaks)
            # Affordable limits for each AMI tier:
            c = 0
            for i in ami_breaks:

                df_all_calcs_full_aff.loc[c, '%s' %(str(year) + '_' + n)] = breaks_dict[i]
                df_all_calcs_full_aff.loc[c+1, '%s' %(str(year) + '_' + n)] = breaks_dict[i]/12
                
                aff_rent = breaks_dict[i]/12

                m = [x for x in l if x <= aff_rent]
                counter = [key for key, value in max_dict.items() if value in m]
                counter = int(df_all_calcs_full[df_all_calcs_full.Attribute.isin(counter)]['%s' %(str(year) + '_' + n)].sum())

                df_all_calcs_full_aff.loc[c+2, '%s' %(str(year) + '_' + n)] = counter    
                
                c+=3
        except Exception as e:
            
            print('aff %s, %s' %(n,e))
        
        try:
            
            median_income = float(df_all_calcs_full_sf[df_all_calcs_full_sf.Attribute.isin([
                                            "Median Household Income"                                                                            
                                            ])].to_dict('records')[0]['%s' %(str(year) + '_' + 'San Francisco')])

            income_breaks = [(i/100) * median_income for i in ami_breaks]
            breaks_dict = dict(zip(ami_breaks, income_breaks))

            #print(breaks_dict, income_breaks)
            # Affordable limits for each AMI tier:
            c = 0
            for i in ami_breaks:

                df_all_calcs_full_sf_aff.loc[c, '%s' %(str(year) + '_' + 'San Francisco')] = breaks_dict[i]
                df_all_calcs_full_sf_aff.loc[c+1, '%s' %(str(year) + '_' + 'San Francisco')] = breaks_dict[i]/12

                aff_rent = breaks_dict[i]/12

                m = [x for x in l if x <= aff_rent]
                counter = [key for key, value in max_dict.items() if value in m]
                counter = int(df_all_calcs_full_sf[df_all_calcs_full_sf.Attribute.isin(counter)]['%s' %(str(year) + '_' + 'San Francisco')].sum())

                df_all_calcs_full_sf_aff.loc[c+2, '%s' %(str(year) + '_' + 'San Francisco')] = counter    
                #print(df_all_calcs_full_sf_aff.loc[c:c+2])
                c+=3

        except Exception as e:
            
            print('aff sf, %s' %(e))

    df_all_calcs_full = pd.concat([df_all_calcs_full, df_all_calcs_full_aff], axis=0)
    df_all_calcs_full_sf = pd.concat([df_all_calcs_full_sf, df_all_calcs_full_sf_aff], axis=0)
    
    year_columns.sort()  
    columns_plus = [str(x) +'_' + n for x in year_columns]
    columns_plus_sf = [str(x) +'_' + 'San Francisco' for x in year_columns]

    # drop na rows
    df_all_calcs_full = df_all_calcs_full.dropna(subset=columns_plus, inplace=False)
    df_all_calcs_full_sf = df_all_calcs_full_sf.dropna(subset=columns_plus_sf, inplace=False)

    columns_plus.insert(0,'Attribute')
    columns_plus.insert(0,'Category')
    columns_plus_sf.insert(0,'Attribute')
    columns_plus_sf.insert(0,'Category')
    
    df_all_calcs_full = df_all_calcs_full[columns_plus]    
    df_all_calcs_full_sf = df_all_calcs_full_sf[columns_plus_sf]    

    alternates = df_all_calcs_full[df_all_calcs_full['Attribute'].str.contains('alternate')]['Attribute'].tolist()
    if len(alternates) > 0:
        alternates_labels = [x[:-11] for x in alternates]
        for a in alternates_labels:
            df_rows = df_all_calcs_full[df_all_calcs_full['Attribute'].str.startswith(a)]
            df_rows = df_rows.iloc[0].combine_first(df_rows.iloc[1])
            df_all_calcs_full = df_all_calcs_full[(df_all_calcs_full['Attribute'].str.startswith(a)) == False]
            df_all_calcs_full = pd.concat([df_all_calcs_full, df_rows.to_frame().T], axis=0).reset_index(drop=True)

    df_all_calcs_full = df_all_calcs_full.sort_values(by= ['Category', 'Attribute'], ascending = [True, True], inplace=False)

    # SF
    alternates = df_all_calcs_full_sf[df_all_calcs_full_sf['Attribute'].str.contains('alternate')]['Attribute'].tolist()
    if len(alternates) > 0:
        alternates_labels = [x[:-11] for x in alternates]
        for a in alternates_labels:
            df_rows = df_all_calcs_full_sf[df_all_calcs_full_sf['Attribute'].str.startswith(a)]
            df_rows = df_rows.iloc[0].combine_first(df_rows.iloc[1])
            df_all_calcs_full_sf = df_all_calcs_full_sf[(df_all_calcs_full_sf['Attribute'].str.startswith(a)) == False]
            df_all_calcs_full_sf = pd.concat([df_all_calcs_full_sf, df_rows.to_frame().T], axis=0).reset_index(drop=True)

    df_all_calcs_full_sf = df_all_calcs_full_sf.sort_values(by= ['Category', 'Attribute'], ascending = [True, True], inplace=False)
    
    year_columns = [2000, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
    comparative_columns = [[str(x) +'_' + n, str(x) +'_' + 'San Francisco'] for x in year_columns]
    comparative_columns = [x for xs in comparative_columns for x in xs ]
    comparative_columns.insert(0,'Attribute')
    comparative_columns.insert(0,'Category')
    
    df_comparative = pd.merge(df_all_calcs_full, df_all_calcs_full_sf, on='Attribute', how='left')
    df_comparative = df_comparative.rename(columns={'Category_x': 'Category'})
    
    try:
        df_comparative = df_comparative[comparative_columns]
    except Exception as e:
        df_comparative = df_comparative
        
    n = n.strip().replace('/','-')
    df_comparative.to_excel(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/output/%s_neighborhood_v_sf_by_attribute_2000to2024.xlsx' %n, index=False, sheet_name='census_attributes')
    df_comparative.to_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/output/%s_neighborhood_v_sf_by_attribute_2000to2024.csv' %n)

## Generate tract-level profiles

In [ ]:
# In[]:
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")
download_path = r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/downloads'
geo_lookup_df = pd.read_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/lookup_tables/geo_lookup_2000to2020.csv', dtype=str)
geo_lookup_df_yyyy = geo_lookup_df[geo_lookup_df['year'] == '2020']
neighborhoods = list(set(geo_lookup_df_yyyy.neighborhood.tolist()))
neighborhoods.sort()

for n in neighborhoods:
    print(n)
    YEARS = [2000, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
    
    tracts = list(set(geo_lookup_df[geo_lookup_df['neighborhood'] == '%s' %n]['tractid'].tolist()))
    tracts.sort()
    TRACTS = []
    for j in tracts:
        if len(j) == 5:
            TRACTS.append('0' + j)  
            
    columns = ['year','label','Category','Attribute'] + TRACTS
    
    df_all_calcs_by_tract_all_years = pd.DataFrame(columns = columns)
    
    for year in YEARS:    
    
        if year <= 2009:
            geo_year = '2000'
            survey = 'dec'
            table_name = 'sf3'
            label = 'DEC_SF3'
            ext = ''
        elif year >= 2010 and year <= 2019:
            geo_year = '2010'
            survey = 'acs'
            table_name = 'acs'
            label = 'ACS5YR'
            ext = 'E'
        elif year >= 2020:
            geo_year = '2020'
            survey = 'acs'
            table_name = 'acs'
            label = 'ACS5YR'
            ext = 'E'
            
        # export both dataset views to csv#
        df_all_calcs_by_tract = pd.read_csv(os.path.join(download_path,'tract'+'_'+'profiles_by_attribute_'+label+'_{}.csv'.format(year)))
        df_all_calcs_by_tract['year'] = year
        df_all_calcs_by_tract['label'] = label
        
        df_all_calcs_by_tract = df_all_calcs_by_tract.sort_values(by=['year','Category', 'Attribute'], \
                                                                  ascending=[True, True, True], inplace=False)
        df_all_calcs_by_tract.reset_index(drop=True)
        
        # CALCULATE AFFORDABILITY
        ami_breaks = [30, 50, 80, 100]
        df_all_calcs_by_tract_aff = pd.DataFrame(columns = df_all_calcs_by_tract.columns, \
                                                index = range(12))
        rent_levels = df_all_calcs_by_tract[(df_all_calcs_by_tract.Attribute.str.contains("Rent")) & \
                           ((df_all_calcs_by_tract.Attribute.str.contains(" to ")) | \
                           (df_all_calcs_by_tract.Attribute.str.contains(" or ")))].to_dict('records')
        max_dict = {}
        for j in rent_levels:
            level_j = j
            key_j = j['Attribute']
            key_j_split = key_j.split(' ')
            try:
                max_dict[key_j] = int(key_j_split[3])
            except:
                max_dict[key_j] = 1e6
        
        l = list(max_dict.values())
        df_all_calcs_by_tract_aff['year'] = int('%s' %year)
        df_all_calcs_by_tract_aff['label'] = '%s' %label
        df_all_calcs_by_tract_aff['Category'] = 'Affordability'
        
        for tract in TRACTS:
            #print(year, tract)
            try:
                median_income = float(df_all_calcs_by_tract[df_all_calcs_by_tract.Attribute.isin([
                                    "Median Household Income"                                                                            
                                    ])].to_dict('records')[0][tract])

                income_breaks = [(i/100) * median_income for i in ami_breaks]
                breaks_dict = dict(zip(ami_breaks, income_breaks))
                # Affordable limits for each AMI tier:
                c = 0
                for i in ami_breaks:
                    
                    df_all_calcs_by_tract_aff.loc[c, 'Attribute'] = '%s AMI Annual Income' %i
                    df_all_calcs_by_tract_aff.loc[c, tract] = breaks_dict[i]
                    df_all_calcs_by_tract_aff.loc[c+1, 'Attribute'] = '%s AMI Monthly Affordable Rent' %i
                    df_all_calcs_by_tract_aff.loc[c+1, tract] = breaks_dict[i]/12
                    df_all_calcs_by_tract_aff.loc[c+2, 'Attribute'] = '# Units at or below %s AMI Monthly Affordable Rent' %i
                    
                    aff_rent = breaks_dict[i]/12
                   
                    m = [x for x in l if x <= aff_rent]
                    counter = [key for key, value in max_dict.items() if value in m]
                    counter = int(df_all_calcs_by_tract[df_all_calcs_by_tract.Attribute.isin(counter)][tract].sum())
                    df_all_calcs_by_tract_aff.loc[c+2, tract] = counter    
                    
                    c+=3
                    
            
            except Exception as e:
                pass
                #print(e)

        df_all_calcs_by_tract = pd.concat([df_all_calcs_by_tract, df_all_calcs_by_tract_aff], axis=0)
        
        for tract in TRACTS:
         
            try:
                df_all_calcs_by_tract[tract]
            except:
                df_all_calcs_by_tract[tract] = 'INACTIVE'
    
        df_all_calcs_by_tract = df_all_calcs_by_tract[columns]
        df_all_calcs_by_tract_all_years = pd.concat([df_all_calcs_by_tract_all_years,df_all_calcs_by_tract], axis=0)
        
    df_all_calcs_by_tract_all_years = df_all_calcs_by_tract_all_years.sort_values(by= ['year', 'Category', 'Attribute'], ascending = [True, True, True], inplace=False)

    n = n.strip().replace('/','-')
    
    df_all_calcs_by_tract_all_years.to_excel(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/output/%s_tract_profiles_by_attribute_2000to2024.xlsx' %n, index=False, sheet_name='census_attributes')
    df_all_calcs_by_tract_all_years.to_csv(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/output/%s_tract_profiles_by_attribute_2000to2024.csv' %n)

## Diagnostic fields 

In [ ]:
'''
import matplotlib.pyplot as plt
YEARS = [2000, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
n = 'San Francisco'
df = pd.read_excel(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/output/%s_neighborhood_profiles_by_attribute_2000to2024.xlsx' %n, sheet_name='census_attributes')
categories = df.Category.tolist()

for c in categories:

    df_cat = df[df['Category'] == c]

    att = df_cat['Attribute'].tolist()

    fig, ax = plt.subplots(figsize=(10, 8))
    
    for a in att:

        df_row = df_cat[df_cat['Attribute'] == a]
        df_row = df_row.drop(['Category', 'Attribute'], axis=1)
        
        ax.plot(YEARS, df_row.iloc[0].values, '-o', label= a)

    ax.legend()
    ax.grid(True)
    ax.set_title(c)
    
    c = c.strip().replace('/','-')

    fig.savefig(r'C:/Users/blatto/Documents/GitHub/socio-economic-profiles-main/socio-economic-profiles/diagnostics/sf_diagnostics_%s.png' %c)
    plt.close(fig)  # if you do not need to leave the figures open
'''